# SP500 Executive Role Classification - Fine-Tuning GPT-OSS-20B

Fine-tunes OpenAI's gpt-oss-20b (21B total / 3.6B active MoE) to classify executive ranks and roles from SP500 proxy filings.

**GPU**: Works on free Colab T4 (16GB) — QLoRA needs ~14GB VRAM.

**Runtime**: Go to Runtime → Change runtime type → Select **T4 GPU** (or A100 for faster training).

## 0. Configuration

Set all key parameters here. Change these before running the rest of the notebook.

In [ ]:
# ============================================================
# CONFIGURATION - Edit these before running
# ============================================================

# GitHub repo
GITHUB_REPO = "https://github.com/daalbert981/finetune_sp500.git"

# Base model — OpenAI gpt-oss-20b (21B total, 3.6B active MoE)
# Pre-quantized by Unsloth for fastest loading
MODEL_NAME = "unsloth/gpt-oss-20b-bnb-4bit"

# HuggingFace Hub settings
HF_REPO_ID = "YOUR_HF_USERNAME/sp500-exec-classifier"  # where to push the model
HF_TOKEN = ""  # paste your HF write token here, or use huggingface-cli login

# Training hyperparameters
TRAIN_VALID_SPLIT = 0.95       # 95% train, 5% validation
NUM_EPOCHS = 3                 # 3-5 for this dataset size
LEARNING_RATE = 2e-4           # standard for QLoRA fine-tuning
BATCH_SIZE = 2                 # keep low for MoE model on T4; increase to 4 on A100
GRADIENT_ACCUMULATION = 8      # effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION = 16
MAX_SEQ_LENGTH = 2048          # instructions + input + output fit well within this
WARMUP_RATIO = 0.03            # % of steps for LR warmup
WEIGHT_DECAY = 0.01
SEED = 42

# LoRA hyperparameters
LORA_R = 64                    # rank — higher = more capacity, more memory
LORA_ALPHA = 16                # scaling factor
LORA_DROPOUT = 0.0             # Unsloth optimized: 0 is fine

# Quantization
LOAD_IN_4BIT = True            # 4-bit quantization (~14GB VRAM for gpt-oss-20b)

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth (handles all dependencies including transformers, peft, trl, bitsandbytes)
!pip install unsloth
# Unsloth auto-detects your GPU and installs the right CUDA version
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → GPU")

## 2. Clone Repo & Load Data

In [ ]:
import os

REPO_DIR = "/content/finetune_sp500"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

DATA_PATH = os.path.join(REPO_DIR, "Training Datasets", "full_data_n_4977.jsonl")
assert os.path.exists(DATA_PATH), f"Data file not found at {DATA_PATH}"

In [ ]:
import json

# Load the clean JSONL (already in chat message format, all 4,977 consistent)
with open(DATA_PATH, "r") as f:
    examples = [json.loads(line) for line in f]

print(f"Total examples: {len(examples)}")

# Verify consistency
sample = examples[0]
print(f"\nMessage structure: {[m['role'] for m in sample['messages']]}")
print(f"System prompt length: {len(sample['messages'][0]['content'])} chars")
print(f"\n--- Sample ---")
print(f"User:      {sample['messages'][1]['content'][:150]}")
print(f"Assistant: {sample['messages'][2]['content']}")

# Sanity check: all have both <rank> and <role>
assert all("<rank>" in e["messages"][2]["content"] and "<role>" in e["messages"][2]["content"] for e in examples), \
    "Not all examples have both <rank> and <role> tags!"
print(f"\nAll {len(examples)} examples have both <rank> and <role> tags.")

## 3. Format Data for Chat Fine-Tuning & Train/Valid Split

In [ ]:
from datasets import Dataset
import random

random.seed(SEED)

# Shuffle and split
random.shuffle(examples)
split_idx = int(len(examples) * TRAIN_VALID_SPLIT)

train_dataset = Dataset.from_list(examples[:split_idx])
valid_dataset = Dataset.from_list(examples[split_idx:])

print(f"Train: {len(train_dataset)} examples")
print(f"Valid: {len(valid_dataset)} examples")
print(f"\nSample formatted message:")
print(f"  User:      {train_dataset[0]['messages'][1]['content'][:120]}")
print(f"  Assistant: {train_dataset[0]['messages'][2]['content']}")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect (float16 for T4, bfloat16 for A100)
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model parameters: {model.num_parameters() / 1e9:.1f}B")

## 5. Configure LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth optimized — saves VRAM
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable / 1e6:.1f}M / {total / 1e9:.1f}B ({100 * trainable / total:.2f}%)")
print(f"GPU VRAM used: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

## 6. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=SFTConfig(
        output_dir="./outputs",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        seed=SEED,
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
    ),
)

print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"Total training steps: ~{len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION)}")

In [ ]:
# GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.max_memory_reserved() / 1e9
print(f"GPU memory reserved before training: {reserved:.2f} GB / {gpu_stats.total_mem / 1e9:.2f} GB")

In [ ]:
# Train!
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Train loss: {train_result.training_loss:.4f}")
print(f"Train runtime: {train_result.metrics['train_runtime'] / 60:.1f} min")
print(f"Peak GPU memory: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")

## 7. Quick Validation Check

In [ ]:
# Run evaluation on validation set
eval_results = trainer.evaluate()
print(f"Validation loss: {eval_results['eval_loss']:.4f}")
print(f"Validation perplexity: {2.71828 ** eval_results['eval_loss']:.2f}")

In [ ]:
# Test inference on a few validation examples
FastLanguageModel.for_inference(model)

num_test = 5
correct = 0

for i in range(min(num_test, len(valid_dataset))):
    messages = valid_dataset[i]["messages"]
    prompt_messages = messages[:2]  # system + user only

    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            temperature=0.0,
            do_sample=False,
        )

    generated = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    expected = messages[2]["content"]

    match = generated.strip() == expected.strip()
    correct += int(match)

    print(f"\n--- Example {i+1} {'✓' if match else '✗'} ---")
    print(f"Input:    {messages[1]['content'][:120]}")
    print(f"Expected: {expected}")
    print(f"Got:      {generated}")

print(f"\nExact match: {correct}/{num_test}")

## 8. Save to HuggingFace Hub

Three save options:
1. **LoRA adapters only** (small, ~100-300MB) — loads on top of base model
2. **Merged 16-bit** (full model, largest) — standalone, no base model needed
3. **Merged 4-bit GGUF** (for llama.cpp / Ollama deployment)

In [ ]:
# Login to HuggingFace
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    print("No HF_TOKEN set. Run: huggingface-cli login")
    !huggingface-cli login

In [ ]:
# Option A: Save LoRA adapters only (recommended for flexibility)
model.save_pretrained_merged(
    "./outputs/merged_model",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved locally.")

In [ ]:
# Push LoRA adapters to HuggingFace Hub
model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN if HF_TOKEN else None,
)
print(f"Model pushed to https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Option B: Also save GGUF for llama.cpp / Ollama (optional)
# Uncomment to save in GGUF format:

# model.push_to_hub_gguf(
#     HF_REPO_ID + "-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token=HF_TOKEN if HF_TOKEN else None,
# )
# print(f"GGUF model pushed to https://huggingface.co/{HF_REPO_ID}-gguf")

## 9. Inference Helper (for Holdout Evaluation)

Use this to generate predictions on your holdout set after training.

In [ ]:
import csv
import re

# The full system instructions for inference
FULL_SYSTEM_PROMPT = """This assistant is trained to code executive ranks and roles along the following categories with 1 or 0.

Ranks:
- VP: 1 if Vice President (VP), 0 otherwise
- SVP: 1 if Senior Vice President (SVP), 0 otherwise
- EVP: 1 if Executive Vice President (EVP), 0 otherwise
- SEVP: 1 if Senior Executive Vice President (SEVP), 0 otherwise
- Director: 1 if Director, 0 otherwise
- Senior Director: 1 if Senior Director, 0 otherwise
- MD: 1 if Managing Director (MD), 0 otherwise
- SMD: 1 if Senior Managing Director (SMD), 0 otherwise
- SE: 1 if Senior Executive, 0 otherwise
- VC: 1 if Vice Chair (VC), 0 otherwise
- SVC: 1 if Senior Vice Chair (SVC), 0 otherwise
- President: 1 if President of the parent company, 0 when President of subsidiary or division but not parent company.

Roles:
- Board: 1 when role suggests person is a member of the board of directors, 0 otherwise
- CEO: 1 when Chief Executive Officer of parent company, 0 when Chief Executive Officer of a subsidiary but not parent company.
- CXO: 1 when C-Suite title, i.e., Chief X Officer, where X can be any type of designation, 0 otherwise. Chief Executive Officer of the parent company. Not Chief AND Officer, e.g., only officer of a function.
- Primary: 1 when responsible for primary activity of value chain, i.e., Supply Chain, Manufacturing, Operations, Marketing & Sales, Customer Service and alike, 0 when not a primary value chain activity.
- Support: 1 when responsible for a support activity of the value chain, i.e., Procurement, IT, HR, Management, Strategy, HR, Finance, Legal, R&D, Investor Relations, Technology, General Counsel and alike, 0 when not support activity of the value.
- BU: 1 when involved with an entity/distinct unit responsible for Product, Customer, or Geographical domain/unit; or role is about a subsidiary, 0 when responsibility is not for a specific product/customer/geography area but, for example, for the entire parent company."""


def predict_single(role_title: str, company: str, year: int, name: str) -> str:
    """Run inference on a single executive."""
    user_msg = (
        f"In {year} the company '{company}' had an executive with the name {name}, "
        f"whose official role title was: '{role_title}'."
    )

    messages = [
        {"role": "system", "content": FULL_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=200, temperature=0.0, do_sample=False
        )

    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()


def parse_output(text: str) -> dict:
    """Parse model output like <rank>vp=0;svp=1;...</rank><role>board=0;...</role> into dict."""
    result = {}
    for tag in ["rank", "role"]:
        match = re.search(f"<{tag}>(.*?)</{tag}>", text)
        if match:
            for pair in match.group(1).split(";"):
                if "=" in pair:
                    k, v = pair.split("=", 1)
                    result[k.strip()] = int(v.strip())
    return result


# Quick test
test_output = predict_single(
    "Senior Vice President, Human Resources",
    "apple inc",
    2020,
    "jane doe"
)
print(f"Raw output: {test_output}")
print(f"Parsed: {parse_output(test_output)}")

In [ ]:
# Batch predict on holdout set
HOLDOUT_PATH = os.path.join(REPO_DIR, "Training Datasets", "holdout_labeling_09072024_template.csv")

with open(HOLDOUT_PATH, "r", encoding="latin-1") as f:
    reader = csv.DictReader(f)
    holdout_rows = list(reader)

print(f"Holdout examples: {len(holdout_rows)}")
print(f"Columns: {list(holdout_rows[0].keys())}")

# Preview first few predictions
for row in holdout_rows[:3]:
    pred = predict_single(row["role"], row["company"], int(row["year"]), row["full_name"])
    parsed = parse_output(pred)
    print(f"\n{row['full_name']} - {row['role']} @ {row['company']}")
    print(f"  Predicted: {parsed}")
    # Compare with ground truth
    truth = {k: int(row[k]) for k in ["vp","svp","evp","sevp","board","ceo","cxo","primary","support","bu","president"] if k in row}
    print(f"  Truth:     {truth}")